In [2]:
import os
env_file = os.getenv("ENV_FILE", "../.env")

import dotenv

In [5]:
AV_API_KEY = dotenv.get_key(env_file, "ALPHAVANTAGE_API_KEY")
assert AV_API_KEY is not None, "ALPHAVANTAGE_API_KEY not found in .env file"

In [6]:
SYMBOL = "ITC"
EXCHANGE = "BSE"

AV_SYMBOL = f"{SYMBOL}.{EXCHANGE}"

# Alpha Vantage — free-tier access for Indian stocks

This notebook surveys which Alpha Vantage endpoints actually work with a **free** API key
for Indian equities (BSE / NSE symbols). The key is loaded from `../.env` in the cells above;
`SYMBOL` / `EXCHANGE` / `AV_SYMBOL` are already defined (`ITC.BSE` by default).

> Verified against the free key on 2026-08-25. Alpha Vantage moves endpoints between the
> free and premium tiers from time to time, so the survey cell at the bottom re-checks
> everything live.


## Indian symbol convention

- **BSE**: `SYMBOL.BSE` (e.g. `ITC.BSE`, `RELIANCE.BSE`, `TATAMOTORS.BSE`) — well supported.
- **NSE**: `SYMBOL.NSE` (e.g. `ITC.NSE`) — spotty coverage; test before relying on it.

The symbol is composed in the existing cell as `f"{SYMBOL}.{EXCHANGE}"`.


## Free-tier rate limits

- **25 requests/day** and roughly **5 requests/minute** (fewer if you fire requests back-to-back).
- The helper below auto-throttles (~13s between calls) so sequential cells stay under the
  per-minute limit. A rate-limited response returns a payload with an `Information`/`Note` key
  rather than the JSON you asked for.


In [ ]:
import time
from typing import Any, Dict, Optional

import requests
import pandas as pd
from IPython.display import display

BASE = "https://www.alphavantage.co/query"
RATE_LIMIT_SECONDS = 13  # free tier ~5 requests/min

_last_call = [0.0]


def av_get(function: str, **params) -> Dict[str, Any]:
    """Call Alpha Vantage, throttling to stay under the free rate limit."""
    wait = RATE_LIMIT_SECONDS - (time.time() - _last_call[0])
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

    payload = {"function": function, "apikey": AV_API_KEY}
    payload.update({k: v for k, v in params.items() if v is not None})
    resp = requests.get(BASE, params=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()


def classify(resp: Dict[str, Any]) -> str:
    """Collapse an Alpha Vantage response into a one-line verdict."""
    if not resp:
        return "EMPTY RESPONSE"
    if "Error Message" in resp:
        return f"ERROR: {resp['Error Message']}"
    if "Note" in resp:
        return f"RATE-LIMITED: {resp['Note']}"
    if "Information" in resp:
        msg = resp["Information"]
        return "PREMIUM" if "premium" in msg.lower() else f"INFO: {msg}"
    if "bestMatches" in resp:
        return "OK (search)"
    if "Global Quote" in resp:
        return "OK (quote)"
    if "Realtime Currency Exchange Rate" in resp:
        return "OK (fx rate)"
    series = [k for k in resp if "Series" in k or k.startswith(("Time Series", "Weekly", "Monthly"))]
    return f"OK ({', '.join(sorted(series)[:2])})" if series else "OK (unexpected keys)"


def series_to_df(resp: Dict[str, Any], key: Optional[str] = None) -> pd.DataFrame:
    """Convert a time-series payload into a DataFrame indexed by date."""
    if key is None:
        key = next((k for k in resp if "Series" in k), None)
    if not key or key not in resp:
        return pd.DataFrame()
    df = pd.DataFrame.from_dict(resp[key], orient="index")
    df.index = pd.to_datetime(df.index)
    return df.astype(float)

## What's free vs premium (verified)

| function | free key | notes |
|---|---|---|
| `TIME_SERIES_DAILY` | ✅ yes | last ~100 sessions |
| `TIME_SERIES_DAILY_ADJUSTED` | ❌ premium | use DAILY / WEEKLY_ADJUSTED |
| `TIME_SERIES_WEEKLY` / `WEEKLY_ADJUSTED` | ✅ yes | |
| `TIME_SERIES_MONTHLY` / `MONTHLY_ADJUSTED` | ✅ yes | |
| `TIME_SERIES_INTRADAY` | ❌ premium | |
| `GLOBAL_QUOTE` | ✅ yes | current price/volume |
| `SYMBOL_SEARCH` | ✅ yes | limited match list |
| `NEWS_SENTIMENT` | ⚠️ verify | often limited on free key |
| `CURRENCY_EXCHANGE_RATE` / `FX_*` | ✅ yes | forex is free |
| `OVERVIEW` / `INCOME_STATEMENT` / `BALANCE_SHEET` | ❌ premium | fundamentals gated |
| `CASH_FLOW` / `EARNINGS` | ❌ premium | fundamentals gated |
| `SMA` / `RSI` / `MACD` … technicals | ⚠️ verify | see survey cell |

Practical takeaway: for Indian stocks on a free key you reliably get **daily / weekly /
monthly** OHLCV, the **current quote**, and **symbol search** — but *not* adjusted-daily,
intraday, or fundamentals.


## End-of-day time series

Each of these returns daily/weekly/monthly OHLCV for `ITC.BSE`.


In [ ]:
r = av_get("TIME_SERIES_DAILY", symbol=AV_SYMBOL)
print(classify(r))
series_to_df(r, "Time Series (Daily)").tail()

In [ ]:
r = av_get("TIME_SERIES_WEEKLY_ADJUSTED", symbol=AV_SYMBOL)
print(classify(r))
series_to_df(r, "Weekly Adjusted Time Series").tail()

In [ ]:
r = av_get("TIME_SERIES_MONTHLY_ADJUSTED", symbol=AV_SYMBOL)
print(classify(r))
series_to_df(r, "Monthly Adjusted Time Series").tail()

## Current quote

`GLOBAL_QUOTE` gives last price, change, % change and volume for the Indian symbol.


In [ ]:
r = av_get("GLOBAL_QUOTE", symbol=AV_SYMBOL)
print(classify(r))
display(r.get("Global Quote", {}))

## Symbol search

Find the exchange suffix / exact symbol. Search on the bare keyword (no `.BSE`).


In [ ]:
r = av_get("SYMBOL_SEARCH", keywords=SYMBOL)
print(classify(r))
pd.DataFrame(r.get("bestMatches", []))

## Forex (USD/INR)

Forex endpoints are not symbol-suffixed and are fully available on the free tier.


In [ ]:
r = av_get("CURRENCY_EXCHANGE_RATE", from_currency="USD", to_currency="INR")
print(classify(r))
display(r.get("Realtime Currency Exchange Rate", {}))

r = av_get("FX_DAILY", from_symbol="USD", to_symbol="INR")
print(classify(r))
series_to_df(r, "Time Series FX (Daily)").tail()

## Premium endpoints (what free keys *don't* get)

These will return an `Information` message saying `premium`, or an empty payload.
Keep an eye on this: Alpha Vantage migrates things like `GLOBAL_QUOTE` between tiers.


In [ ]:
for func, params in [
    ("TIME_SERIES_DAILY_ADJUSTED", {"symbol": AV_SYMBOL}),
    ("TIME_SERIES_INTRADAY", {"symbol": AV_SYMBOL, "interval": "5min"}),
    ("OVERVIEW", {"symbol": AV_SYMBOL}),
    ("INCOME_STATEMENT", {"symbol": AV_SYMBOL}),
]:
    print(f"{func:28} -> {classify(av_get(func, **params))}")

## Full live survey (optional, ~4 min)

Re-verifies every endpoint above in one pass. Throttled, so it takes ~13s per row.


In [ ]:
SURVEY = [
    ("TIME_SERIES_DAILY", {"symbol": "ITC.BSE"}),
    ("TIME_SERIES_DAILY_ADJUSTED", {"symbol": "ITC.BSE"}),
    ("TIME_SERIES_WEEKLY_ADJUSTED", {"symbol": "ITC.BSE"}),
    ("TIME_SERIES_MONTHLY_ADJUSTED", {"symbol": "ITC.BSE"}),
    ("TIME_SERIES_INTRADAY", {"symbol": "ITC.BSE", "interval": "5min"}),
    ("TIME_SERIES_DAILY", {"symbol": "ITC.NSE"}),
    ("GLOBAL_QUOTE", {"symbol": "ITC.BSE"}),
    ("SYMBOL_SEARCH", {"keywords": "ITC"}),
    ("NEWS_SENTIMENT", {"tickers": "ITC"}),
    ("CURRENCY_EXCHANGE_RATE", {"from_currency": "USD", "to_currency": "INR"}),
    ("FX_DAILY", {"from_symbol": "USD", "to_symbol": "INR"}),
    ("OVERVIEW", {"symbol": "ITC.BSE"}),
    ("INCOME_STATEMENT", {"symbol": "ITC.BSE"}),
    ("BALANCE_SHEET", {"symbol": "ITC.BSE"}),
    ("CASH_FLOW", {"symbol": "ITC.BSE"}),
    ("EARNINGS", {"symbol": "ITC.BSE"}),
    ("SMA", {"symbol": "ITC.BSE", "interval": "daily", "time_period": 10, "series_type": "close"}),
    ("RSI", {"symbol": "ITC.BSE", "interval": "daily", "time_period": 14, "series_type": "close"}),
]

rows = []
for func, params in SURVEY:
    try:
        rows.append({"function": func, "params": str(params), "result": classify(av_get(func, **params))})
    except Exception as exc:  # noqa: BLE001
        rows.append({"function": func, "params": str(params), "result": f"EXCEPTION: {exc}"})

pd.DataFrame(rows)